# DNA → RNA → Asam Amino

Tugas Kelompok 1 · Bioinformatika (Pertemuan 7) · UGM

**Kelompok / anggota:** _(isi nama dan NIM)_

Notebook ini mengubah urutan **DNA** menjadi **urutan asam amino (protein)** dengan melewati tahap **mRNA** terlebih dahulu (dogma sentral: DNA → RNA → protein). Tidak perlu memahami kodenya dan tidak ada yang perlu dipasang.

## Cara pakai (3 langkah)

1. Isi kotak **"1. Masukkan DNA di sini"** di bawah. Nilai yang sudah terisi adalah contoh dari slide kuliah.
2. Klik menu **Runtime → Run all** (Jalankan semua), atau tekan `Ctrl + F9`.
3. Hasilnya muncul di bawah kotak **"3. Hasil"**.

Untuk mencoba DNA lain, ubah isiannya lalu ulangi langkah 2.

**Format DNA yang diterima.** Huruf besar atau kecil, dengan atau tanpa spasi, angka, dan baris baru, termasuk hasil salin dari FASTA (baris judul yang diawali `>` diabaikan). Isinya hanya boleh huruf **A, C, G, T**.

## Arti tiap isian

| Isian | Arti |
|---|---|
| `dna` | Urutan DNA yang ingin diubah. Dianggap sebagai untai cetakan: komplemennya dibuat dulu, lalu menjadi mRNA (lihat "Cara kerja"). |
| `berhenti_di_stop` | Dicentang: translasi berhenti pada kodon stop pertama (UAA, UAG, UGA) dan kodon stop tidak ikut ditulis. Tidak dicentang: seluruh urutan diterjemahkan dan stop ditandai `*`. |
| `reading_frame` | Mulai membaca kodon dari basa ke-1, ke-2, atau ke-3 pada mRNA. |
| `tampilkan_tabel_kodon` | Dicentang: tampilkan juga tabel kode genetik (64 kodon). |

In [1]:
#@title 1. Masukkan DNA di sini { display-mode: "form" }

dna = "TCGGTGAATCTGTTTGAT"      #@param {type:"string"}
berhenti_di_stop = False        #@param {type:"boolean"}
reading_frame = 1               #@param [1, 2, 3] {type:"raw"}
tampilkan_tabel_kodon = False   #@param {type:"boolean"}

In [2]:
#@title 2. Algoritma (tidak perlu diubah) { display-mode: "form" }
import itertools
import re


class InputError(ValueError):
    """Kesalahan input; pesannya ditulis untuk pengguna awam."""


# --- Kode genetik: 64 kodon -> 20 asam amino + stop ('*') ------------------
_URUTAN = "TCAG"
_TABEL = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
KODON = {"".join(k): aa for k, aa in zip(itertools.product(_URUTAN, repeat=3), _TABEL)}

NAMA_AA = {
    "A": ("Ala", "Alanine"),        "C": ("Cys", "Cysteine"),
    "D": ("Asp", "Aspartic acid"),  "E": ("Glu", "Glutamic acid"),
    "F": ("Phe", "Phenylalanine"),  "G": ("Gly", "Glycine"),
    "H": ("His", "Histidine"),      "I": ("Ile", "Isoleucine"),
    "K": ("Lys", "Lysine"),         "L": ("Leu", "Leucine"),
    "M": ("Met", "Methionine"),     "N": ("Asn", "Asparagine"),
    "P": ("Pro", "Proline"),        "Q": ("Gln", "Glutamine"),
    "R": ("Arg", "Arginine"),       "S": ("Ser", "Serine"),
    "T": ("Thr", "Threonine"),      "V": ("Val", "Valine"),
    "W": ("Trp", "Tryptophan"),     "Y": ("Tyr", "Tyrosine"),
    "*": ("Stop", "Stop"),
}

_KOMPLEMEN = str.maketrans("ACGT", "TGCA")


# --- 1) Membersihkan dan memeriksa input ------------------------------------
def bersihkan_dna(teks):
    if teks is None or not str(teks).strip():
        raise InputError("Kotak DNA masih kosong. Isi dengan urutan DNA, "
                         "contoh: TCGGTGAATCTGTTTGAT")
    baris = [b for b in str(teks).splitlines() if not b.lstrip().startswith(">")]
    s = re.sub(r"[\s\d]", "", "".join(baris)).upper()
    if not s:
        raise InputError("Tidak ada huruf DNA yang ditemukan (hanya judul FASTA, "
                         "angka, atau spasi).")
    asing = sorted(set(s) - set("ACGT"))
    if asing:
        if asing == ["U"]:
            raise InputError("Urutan ini mengandung huruf U, jadi tampaknya sudah RNA. "
                             "Notebook ini menerima DNA: tulis T sebagai ganti U.")
        daftar = ", ".join(f"'{c}'" for c in asing)
        pertama = min(s.index(c) for c in asing) + 1
        raise InputError(f"Ditemukan karakter selain A, C, G, T: {daftar} "
                         f"(yang pertama pada huruf ke-{pertama}, setelah spasi dan angka "
                         f"dibuang). Periksa kembali urutan DNA.")
    return s


# --- 2) Transkripsi: DNA -> mRNA --------------------------------------------
def transkripsi(dna):
    """Komplemen tiap basa (A<->T, C<->G), lalu T -> U (tanpa dibalik, seperti slide)."""
    komplemen = dna.translate(_KOMPLEMEN)
    return komplemen, komplemen.replace("T", "U")


# --- 3) Translasi: mRNA -> asam amino ---------------------------------------
def ubah_dna_ke_aa(teks_dna, berhenti_di_stop=False, frame=1):
    dna = bersihkan_dna(teks_dna)
    try:
        frame = int(frame)
    except (TypeError, ValueError):
        frame = 0
    if frame not in (1, 2, 3):
        raise InputError("reading_frame harus 1, 2, atau 3.")

    komplemen, mrna = transkripsi(dna)
    bacaan = mrna[frame - 1:]
    jumlah = len(bacaan) // 3
    if jumlah == 0:
        raise InputError("DNA terlalu pendek: minimal 3 basa (setelah reading frame "
                         "diterapkan) untuk membentuk satu kodon.")

    kodon, protein, posisi_stop, berhenti_pada = [], [], [], None
    for i in range(jumlah):
        k = bacaan[3 * i: 3 * i + 3]
        aa = KODON[k.replace("U", "T")]
        kodon.append((k, aa))
        if aa == "*":
            posisi_stop.append(i + 1)
            if berhenti_di_stop:
                berhenti_pada = i + 1
                break
        protein.append(aa)

    return {
        "dna": dna, "frame": frame, "komplemen": komplemen,
        "mrna": mrna, "kodon": kodon, "protein": "".join(protein),
        "posisi_stop": posisi_stop, "berhenti_pada": berhenti_pada,
        "sisa": "" if berhenti_pada else bacaan[3 * jumlah:],
    }


# --- 4) Menampilkan hasil ---------------------------------------------------
def _bungkus(teks, lebar=60):
    if not teks:
        return "(kosong)"
    return "\n".join(teks[i:i + lebar] for i in range(0, len(teks), lebar))


def _tiga_huruf(protein):
    return "-".join(NAMA_AA[a][0] for a in protein)


def tampilkan_hasil(h):
    garis = "=" * 64
    print(garis)
    print("  HASIL:  DNA  ->  mRNA  ->  ASAM AMINO")
    print(garis)
    print(f"DNA: {len(h['dna'])} basa | reading frame {h['frame']}")

    print("\nLANGKAH 1 - Transkripsi (DNA -> mRNA)")
    print("\nDNA yang dimasukkan:")
    print(_bungkus(h["dna"]))
    print("\nKomplemennya (A<->T, C<->G):")
    print(_bungkus(h["komplemen"]))
    print("\nmRNA (T diganti U):")
    print(_bungkus(h["mrna"]))

    print("\nLANGKAH 2 - Translasi (mRNA dibaca per 3 basa = 1 kodon)")
    if h["frame"] > 1:
        print(f"({h['frame'] - 1} basa pertama dilewati: reading frame {h['frame']})")
    BATAS, PER_BARIS = 600, 15
    tampil = h["kodon"][:BATAS]
    for i in range(0, len(tampil), PER_BARIS):
        potong = tampil[i:i + PER_BARIS]
        print()
        print("kodon :", " ".join(k.ljust(3) for k, _ in potong))
        print("asam  :", " ".join(a.ljust(3) for _, a in potong))
    if len(h["kodon"]) > BATAS:
        print(f"\n... {len(h['kodon']) - BATAS} kodon lainnya tidak ditampilkan di sini "
              f"(tetap ada di hasil akhir).")

    p = h["protein"]
    print("\n" + garis)
    print(f"  HASIL AKHIR: {len(p)} asam amino")
    print(garis)
    print("Satu huruf:")
    print(_bungkus(p))
    print("\nTiga huruf:")
    if p:
        per_baris = 15
        for i in range(0, len(p), per_baris):
            print(_tiga_huruf(p[i:i + per_baris]))
    else:
        print("(kosong)")

    if p:
        unik = []
        for a in p:
            if a not in unik:
                unik.append(a)
        print("\nNama asam amino yang muncul:")
        for a in unik:
            if a == "*":
                print("  *  Stop (kodon penghenti, tidak menghasilkan asam amino)")
            else:
                print(f"  {a}  {NAMA_AA[a][0]}  {NAMA_AA[a][1]}")

    catatan = []
    if h["berhenti_pada"]:
        k = h["kodon"][h["berhenti_pada"] - 1][0]
        catatan.append(f"Translasi berhenti di kodon stop {k} (kodon ke-{h['berhenti_pada']}); "
                       f"kodon stop tidak ditulis pada protein.")
    elif h["posisi_stop"]:
        daftar = ", ".join(str(x) for x in h["posisi_stop"][:10])
        catatan.append(f"Ada kodon stop (*) pada kodon ke-{daftar}. Pada makhluk hidup, "
                       f"translasi berhenti di stop pertama; centang 'berhenti_di_stop' "
                       f"untuk memotong protein di titik itu.")
    if h["sisa"]:
        catatan.append(f"Sisa {len(h['sisa'])} basa di ujung ({h['sisa']}) tidak cukup "
                       f"membentuk satu kodon, sehingga tidak diterjemahkan.")
    if catatan:
        print("\nCatatan:")
        for c in catatan:
            print(" -", c)


def tabel_kodon():
    print("\nTABEL KODE GENETIK (kodon mRNA -> asam amino)\n")
    huruf = "UCAG"
    print("1st |" + "".join(f"        {b}        " for b in huruf) + "| 3rd")
    print("-" * 80)
    for pertama in huruf:
        for ketiga in huruf:
            sel = []
            for kedua in huruf:
                k = pertama + kedua + ketiga
                aa = KODON[k.replace("U", "T")]
                sel.append(f"{k} {NAMA_AA[aa][0]:<4}".ljust(18))
            print(f" {pertama if ketiga == 'U' else ' '}  |" + "".join(sel) + f"|  {ketiga}")
        print("-" * 80)

In [3]:
#@title 3. Hasil { display-mode: "form" }
try:
    hasil = ubah_dna_ke_aa(dna, berhenti_di_stop=berhenti_di_stop, frame=reading_frame)
    tampilkan_hasil(hasil)
    if tampilkan_tabel_kodon:
        tabel_kodon()
except InputError as e:
    print("PERHATIAN:", e)
except NameError:
    print("PERHATIAN: sel 1 dan 2 belum dijalankan. Klik Runtime -> Run all.")

  HASIL:  DNA  ->  mRNA  ->  ASAM AMINO
DNA: 18 basa | reading frame 1

LANGKAH 1 - Transkripsi (DNA -> mRNA)

DNA yang dimasukkan:
TCGGTGAATCTGTTTGAT

Komplemennya (A<->T, C<->G):
AGCCACTTAGACAAACTA

mRNA (T diganti U):
AGCCACUUAGACAAACUA

LANGKAH 2 - Translasi (mRNA dibaca per 3 basa = 1 kodon)

kodon : AGC CAC UUA GAC AAA CUA
asam  : S   H   L   D   K   L  

  HASIL AKHIR: 6 asam amino
Satu huruf:
SHLDKL

Tiga huruf:
Ser-His-Leu-Asp-Lys-Leu

Nama asam amino yang muncul:
  S  Ser  Serine
  H  His  Histidine
  L  Leu  Leucine
  D  Asp  Aspartic acid
  K  Lys  Lysine


## Cara kerja (ringkas)

**1. Transkripsi (DNA → mRNA).** DNA yang dimasukkan dibaca sebagai untai cetakan: tiap basa diganti pasangannya (A↔T, C↔G), lalu T diganti U karena RNA memakai Urasil. Mengikuti slide, urutan tidak dibalik.

Contoh dari slide: `GGATGCCAATG` → `CCTACGGTTAC` → `CCUACGGUUAC`.

**2. Translasi (mRNA → asam amino).** mRNA dibaca per tiga basa (kodon). Ada 64 kodon yang menyandi 20 asam amino ditambah sinyal stop (`UAA`, `UAG`, `UGA`). Beberapa kodon menyandi asam amino yang sama, sehingga kode genetik disebut redundan.

Contoh dari slide (nilai awal di sel 1): `TCGGTGAATCTGTTTGAT` → `AGCCACUUAGACAAACUA` → `AGC CAC UUA GAC AAA CUA` → `S H L D K L`.

**3. Reading frame.** Titik awal pembacaan menentukan pengelompokan kodon. Menggeser awal pembacaan satu basa mengubah seluruh kodon sesudahnya dan menghasilkan protein yang sama sekali berbeda.

**Kode asam amino:** A Ala Alanine · C Cys Cysteine · D Asp Aspartic acid · E Glu Glutamic acid · F Phe Phenylalanine · G Gly Glycine · H His Histidine · I Ile Isoleucine · K Lys Lysine · L Leu Leucine · M Met Methionine · N Asn Asparagine · P Pro Proline · Q Gln Glutamine · R Arg Arginine · S Ser Serine · T Thr Threonine · V Val Valine · W Trp Tryptophan · Y Tyr Tyrosine